# Building GPT

## 1. 准备数据

下载数据并查看

In [6]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://ghproxy.net/https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-04 19:57:04--  https://ghproxy.net/https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving ghproxy.net (ghproxy.net)... 51.195.241.253
Connecting to ghproxy.net (ghproxy.net)|51.195.241.253|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M   246KB/s    in 4.4s    

2026-03-04 19:57:16 (246 KB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [15]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [16]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [17]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



数据处理，采用最简单的编解码方式，逐个字符对应，共65个字符；



In [18]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))  # set去重/形成列表/排序
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


也有其他的token处理方式，基于句子或者基于单词，如GPT实际使用的fast BPE tokenizer，例如：

In [19]:
import tiktoken
tik_encoder = tiktoken.get_encoding("gpt2")
print(tik_encoder.encode("hello world"))
print(tik_encoder.decode(tik_encoder.encode("hello world")))

[31373, 995]
hello world


In [20]:
# 创建字符和数字的映射关系，由此构建编码和解码器
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]  # 看往后预测一个元素

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

划分为block，范围内的作为同一批次训练样本，基于自回归可以通过已知tokens预测下一token，再将其作为已知tokens继续预测，如：

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


### 数据加载


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [ ]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


## 模型

1. 需要reshape，因为cross_entropy要求输入是二维的，第一维是样本数量，第二维是类别数量，而现在的logits是三维的，第一维是批量大小，第二维是时间步，第三维是类别数量，所以我们需要把前两维合并成一维，这样每个时间步的预测都被视为一个独立的样本。
2. 监督学习需要有对应的targets来进行损失的计算

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # 转化为一个(vocab_size, vocab_size)的矩阵，行是输入字符，列是输出字符，值是对应的logit
        # 我们要预测下一个字符的概率分布，而这个分布的维度就是vocab_size，所以每个输入字符都对应一个长度为vocab_size的向量，表示对每个可能输出字符的logit值。
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)  # 
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # 接受idx和max_new_tokens参数，idx是当前上下文的索引数组，max_new_tokens是要生成的新令牌的数量
    def generate(self, idx, max_new_tokens):
        # idx是一个(B, T)的张量，表示当前上下文的索引数组，B是批量大小，T是当前上下文的长度，也可以认为是时间步
        for _ in range(max_new_tokens):
            # 调用前向方法获取当前上下文的logits和损失
            logits, loss = self(idx)
            # 只关注最后一个时间步的logits，因为只预测下一个token
            logits = logits[:, -1, :] # becomes (B, C)
            # 转换为概率
            probs = F.softmax(logits, dim=-1) # (B, C)
            # 从概率分布中采样一个索引作为下一个token
            # 为什么采样？因为我们希望生成的文本具有多样性，而不是每次都选择概率最高的令牌，这样可以避免生成重复和无趣的文本。
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # 将新生成的索引追加到当前上下文的idx中，形成新的上下文，以便在下一次迭代中使用
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# 0作为起始token，生成100个新token；索引0取batch的第0个样本；tolist将生成的索引列表转换为Python列表；decode将索引列表转换为字符串
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    # 上一步梯度清零，set_to_none=True可以稍微加速训练；获取参数梯度；更新参数
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

NameError: name 'get_batch' is not defined

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

NameError: name 'decode' is not defined

## 3. 自注意力

自注意力的数学trick
1）预测下一个token需要与前面的tokens进行交互，可以采取求和的方式

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

NameError: name 'torch' is not defined

In [ ]:
xbow = torch.zeros((B, T, C))  # 初始化为0
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]  # (t, C) 当前时间步之前的所有输入
        xbow[b, t] = torch.mean(xprev, dim=0)  # 计算当前时间步之前的所有输入的平均值